# Notebook 02: Simulation Figures — Reproducing Paper Results

**Authors:** Neha Abin, Sahil Shah, Yajat Parmar  
**Affiliation:** Allen High School, Allen TX

---

This notebook reproduces all three key figures from the IEEE paper in Python/matplotlib:

1. **Figure 1:** CCI surface plot vs. velocity and delay (3D surface)
2. **Figure 2:** 3D scatter with threshold plane at γ = 0.6
3. **Figure 3:** Two-panel stability analysis (CCI + perturbation error vs. velocity)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from src.cci import (
    doppler_frequency, coherence_time, channel_autocorrelation,
    condition_number, compute_cci, DEFAULT_GAMMA
)
from src.channel_model import ChannelConfig, simulate_channel_sequence, add_csi_error
from src.beamforming import zf_precoder, mrt_precoder, precoding_error

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12})

COLOR_STABLE = '#3b82f6'
COLOR_THRESHOLD = '#f59e0b'
COLOR_UNSTABLE = '#ef4444'
COLOR_WAVELYNK = '#22c55e'

os.makedirs('../assets/notebooks', exist_ok=True)
print('Setup complete.')

## Figure 1: CCI Surface vs. Velocity and Delay

This 3D surface shows how CCI varies as a function of user velocity (0–50 m/s)
and CSI feedback delay (0–10 ms). The color represents CCI magnitude:
blue = low/stable → yellow = high/unstable.

In [ ]:
# Parameter grid
velocities = np.linspace(0.5, 50, 80)
delays = np.linspace(0.0001, 0.010, 80)  # 0 to 10 ms
V, D = np.meshgrid(velocities, delays)

fc = 6e9
sinr_db = 20.0
alpha, beta = 1.0, 1.0

# Generate CCI surface with parametric κ
# Use condition number that scales mildly with velocity to simulate
# realistic channel degradation
CCI_surface = np.zeros_like(V)

rng = np.random.default_rng(42)
H_base = (rng.standard_normal((4, 4)) + 1j * rng.standard_normal((4, 4))) / np.sqrt(8)

for i in range(V.shape[0]):
    for j in range(V.shape[1]):
        v = V[i, j]
        tau = D[i, j]
        # Parametric κ: increases with velocity (higher mobility → harder channel)
        kappa_scale = 1 + (v / 50) * 19  # κ from 1 to 20
        fD = doppler_frequency(v, fc)
        Tc = coherence_time(fD)
        rho = abs(channel_autocorrelation(fD, tau))
        sinr_lin = 10**(sinr_db / 10)
        sinr_w = sinr_lin / (sinr_lin + alpha)
        aging = np.exp(-beta * tau / Tc) if Tc != np.inf else 1.0
        CCI_surface[i, j] = kappa_scale * rho * sinr_w * aging

# Clip for visualization
CCI_surface = np.clip(CCI_surface, 0, 15)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    V, D * 1000, CCI_surface,  # convert delay to ms
    cmap='viridis', alpha=0.85, edgecolor='none',
    antialiased=True
)

# Add threshold plane at γ = 0.6
xx, yy = np.meshgrid(np.linspace(0.5, 50, 10), np.linspace(0, 10, 10))
zz = np.full_like(xx, DEFAULT_GAMMA)
ax.plot_surface(xx, yy, zz, alpha=0.15, color=COLOR_UNSTABLE)

ax.set_xlabel('User Velocity (m/s)', labelpad=10)
ax.set_ylabel('CSI Delay τ (ms)', labelpad=10)
ax.set_zlabel('CCI Value', labelpad=10)
ax.set_title('Conditioned Coherence Index vs. Velocity and Delay', pad=20)

fig.colorbar(surf, shrink=0.5, aspect=15, label='CCI Magnitude')
ax.view_init(elev=25, azim=135)

plt.tight_layout()
plt.savefig('../assets/notebooks/fig_05_cci_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_05_cci_surface.png')

## Figure 2: 3D Scatter with Threshold Plane

500 random simulation points plotted in (velocity, correlation, CCI) space.
A semi-transparent plane at z = 0.6 shows the switching threshold.

In [ ]:
# Generate 500 random simulation points
rng = np.random.default_rng(2025)
N_points = 500

fc = 6e9
alpha, beta = 1.0, 1.0

scatter_v = rng.uniform(0.5, 45, N_points)
scatter_tau = rng.uniform(0.0005, 0.010, N_points)
scatter_sinr = rng.uniform(5, 30, N_points)

scatter_rho = []
scatter_cci = []

for i in range(N_points):
    v = scatter_v[i]
    tau = scatter_tau[i]
    sinr = scatter_sinr[i]
    
    fD = doppler_frequency(v, fc)
    Tc = coherence_time(fD)
    rho = abs(channel_autocorrelation(fD, tau))
    sinr_lin = 10**(sinr / 10)
    sinr_w = sinr_lin / (sinr_lin + alpha)
    aging = np.exp(-beta * tau / Tc) if Tc != np.inf else 1.0
    
    # Random condition number 1-15
    kappa = rng.uniform(1, 15)
    cci = kappa * rho * sinr_w * aging
    
    scatter_rho.append(rho)
    scatter_cci.append(min(cci, 15))  # clip for viz

scatter_rho = np.array(scatter_rho)
scatter_cci = np.array(scatter_cci)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter points, colored by CCI
sc = ax.scatter(
    scatter_v, scatter_rho, scatter_cci,
    c=scatter_cci, cmap='viridis', s=12, alpha=0.6, edgecolors='none'
)

# Threshold plane at z = 0.6
xx, yy = np.meshgrid(np.linspace(0, 45, 10), np.linspace(0, 1, 10))
zz = np.full_like(xx, DEFAULT_GAMMA)
ax.plot_surface(xx, yy, zz, alpha=0.2, color=COLOR_UNSTABLE, label='γ = 0.6')
ax.text(22, 0.5, DEFAULT_GAMMA + 0.3, 'Switching Threshold γ = 0.6',
        color=COLOR_UNSTABLE, fontsize=11, ha='center', fontweight='bold')

ax.set_xlabel('User Velocity (m/s)', labelpad=10)
ax.set_ylabel('Channel Correlation ρ', labelpad=10)
ax.set_zlabel('CCI Value', labelpad=10)
ax.set_title('CCI vs. Velocity and Channel Correlation — 500 Random Trials', pad=20)

fig.colorbar(sc, shrink=0.5, aspect=15, label='CCI Magnitude')
ax.view_init(elev=20, azim=130)

plt.tight_layout()
plt.savefig('../assets/notebooks/fig_06_3d_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_06_3d_scatter.png')

## Figure 3: Two-Panel Stability Analysis

**Top panel:** CCI vs. velocity with the stability threshold γ = 0.6 marked.

**Bottom panel:** Perturbation error ‖ΔW‖/‖W‖ for three strategies:
- Pure ZF (catastrophic failure past the cliff)
- Pure MRT (robust baseline)
- WaveLynk adaptive switching (stays near MRT after switch point)

In [ ]:
# Simulate channel over velocity sweep
velocities = np.linspace(0.5, 45, 200)
fc = 6e9
tau = 0.005  # 5 ms delay
sinr_db = 20.0
alpha, beta = 1.0, 1.0
dt = 0.001

rng = np.random.default_rng(42)

cci_vs_v = []
error_zf = []
error_mrt = []
error_adaptive = []

for v in velocities:
    # Generate true and delayed channel
    config = ChannelConfig(Nr=4, Nt=4, N_paths=8, carrier_freq_hz=fc,
                          velocity_mps=v, snr_db=sinr_db, seed=42)
    times, H_seq = simulate_channel_sequence(config, duration_s=0.05, dt_s=dt)
    
    # True channel at last timestep
    H_true = H_seq[-1]
    delay_samples = int(tau / dt)
    H_est = add_csi_error(H_true, sinr_db, delay_samples, H_seq)
    
    # Compute CCI
    cci = compute_cci(H_true, v, fc, tau, sinr_db, alpha, beta)
    cci_vs_v.append(cci)
    
    # ZF precoding error (using true vs estimated channel)
    W_zf_true = zf_precoder(H_true)
    W_zf_est = zf_precoder(H_est)
    error_zf.append(precoding_error(W_zf_true, W_zf_est))
    
    # MRT precoding error
    W_mrt_true = mrt_precoder(H_true)
    W_mrt_est = mrt_precoder(H_est)
    error_mrt.append(precoding_error(W_mrt_true, W_mrt_est))
    
    # Adaptive: use ZF when CCI < γ, MRT otherwise
    if cci < DEFAULT_GAMMA:
        error_adaptive.append(error_zf[-1])
    else:
        error_adaptive.append(error_mrt[-1])

# Plotting
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.2]})

# Top panel: CCI vs. velocity
ax = axes[0]
ax.plot(velocities, cci_vs_v, color=COLOR_STABLE, linewidth=2.5, label='CCI')
ax.axhline(DEFAULT_GAMMA, color=COLOR_THRESHOLD, linestyle='--', linewidth=2,
          label=f'Stability Threshold γ = {DEFAULT_GAMMA}')
ax.fill_between(velocities, DEFAULT_GAMMA, max(cci_vs_v) * 1.1,
                where=[c >= DEFAULT_GAMMA for c in cci_vs_v],
                alpha=0.15, color=COLOR_UNSTABLE)
ax.set_ylabel('CCI Value')
ax.set_title('CCI and Precoding Error vs. User Velocity')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Bottom panel: Perturbation error (log scale)
ax = axes[1]
ax.plot(velocities, error_zf, color=COLOR_UNSTABLE, linewidth=2,
       label='Pure ZF (Catastrophic Failure)', linestyle='-')
ax.plot(velocities, error_mrt, color=COLOR_STABLE, linewidth=2,
       label='Pure MRT (Robust Baseline)', linestyle='--')
ax.plot(velocities, error_adaptive, color=COLOR_WAVELYNK, linewidth=2.5,
       label='Proposed Adaptive Switching', linestyle='-')

ax.set_yscale('log')
ax.set_xlabel('User Velocity (m/s)')
ax.set_ylabel('Perturbation Error ‖ΔW‖/‖W‖ (log)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../assets/notebooks/fig_07_stability_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_07_stability_analysis.png')

## Summary

These three figures demonstrate the core findings of the WaveLynk paper:

1. **CCI surface** shows the instability region grows with both velocity and delay
2. **3D scatter** visualizes where individual channel realizations fall relative to the threshold
3. **Stability analysis** proves that WaveLynk adaptive switching avoids the catastrophic ZF failure mode while maintaining comparable performance to MRT

Proceed to `03_monte_carlo.ipynb` for the full 100-trial statistical comparison.